# <center>Rocket Lab Dados - 2026.2</center>
# <center> Introdução à Pyspark</center>
___

# <center> Objetivos de aprendizado </center>
- Familiarizar-se com as funcionalidades básicas do PySpark
- Ser capaz de carregar dados em um DataFrame
- Ser capaz de realizar manipulações básicas de dados
___


In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

#Criar sessão spark
spark = SparkSession.builder.appName("AtividadePraticaSpark").getOrCreate()

## Exercício 1
O Ultimate Team (FUT) é um modo do jogo FIFA no qual o jogador monta seu próprio time adquirindo atletas virtuais.
Cada atleta possui atributos que influenciam seu desempenho em campo — como drible, chute, passe, defesa, velocidade e físico.

Principais colunas:
- `player_id` — identificador único do jogador
- `player_name` — nome do atleta
- `nationality` — país de origem
- `club` — clube atual
- `overall` — nota geral do jogador
- `potential` — potencial máximo de evolução
- `value_eur, wage_eur` — valor de mercado e salário
- `age, height_cm, weight_kg` — características físicas
- `pace, shooting, passing, dribbling, defending, physic` — atributos técnicos

_**Preencha os espacos ____ para carregar os dados e realizar as consultas propostas.**_

### Exercício 1.1 Faça a leitura do arquivo fut_players (fut_player_data.csv) e retorne as 5 primeiras linhas

In [0]:
# Faça a leitura do arquivo fut_players 
fut_players = spark.read.csv(
    "/Workspace/Users/jhonatasousa288@gmail.com/rocket-lab-2026/solucoes/pratica-001/fut_players_data.csv", 
    header=True,
    inferSchema=True)

# Retorne as 5 primeiras linhas do DF
display(fut_players.limit(5))

### Exercício 1.2 - Retorna a nacionalidade dos jogadores "The Bests"

São considerados jogadores The Bests os que possuem os atributos de drible (_dribbling_) e chute (_shooting_) superior a 90. 
Após a geração do DF _The_Best_ realize o join com o df _nationalities_ para obter a nacionalidade dos jogadores.

A sua tabela final deve conter as seguintes informações:
- `player_id`
- `player_name`
- `nationality`
- `position`
- `dribbling`
- `shooting`
- `overall`

In [0]:
# Aplique os filtros para retornar os jogadores the bests
the_best = fut_players.filter((fut_players.dribbling > 90) & (fut_players.shooting > 90))

# nationalities é um DataDrame da nacionalidade dos jogadores
nationalities = (fut_players.select('player_id', 'player_name', 'nationality'))

# faça um join dos dois DataDrames, mantendo todos os jogadores de the_best e obtendo suas nacionalidades (dica: a chave é o id)
the_best_nationality = the_best.join(nationalities, the_best.player_id == nationalities.player_id, "left")

the_best_nationality.display()



###  Exercício 2
Use o método *.groupby()* para descobrir qual país tem o melhor *overall* médio. Crie a coluna 'avg_overall'

Seu df country_avg_overall deve conter as seguintes colunas:
- `nationality`
- `overall`
- `avg_overall`

In [0]:
country_avg_overall = (fut_players.groupBy("nationality").agg(F.avg(F.col("overall")).alias("avg_overall")))

# Retornar a nacionalidade com maior overall médio e o overall médio do brasil
melhor = (
    country_avg_overall
    .orderBy(F.col("avg_overall").desc())
    .limit(1)
    .collect()[0]
)

brasil = (
    country_avg_overall
    .filter(F.col("nationality") == "Brazil")
    .collect()[0]
)

display({
    "Melhor overall médio": f"{melhor['nationality']}: {melhor['avg_overall']:.2f}",
    "Overall médio do Brasil": round(brasil['avg_overall'], 2)
})

Agora nós já cobrimos toda a parte básica do Spark! Vamos praticar essa última parte!

### Exercício 2.1
Crie um racional que retorne a classificação para o jogador de acordo com as instruções abaixo, então aplique isso para o dataframe fut_players.

*Observação:* considere os limites dentro do intervalo de classificação.
exemplo

-50 contém todos os valores menores que 50 e o valor 50 incluso;


51-60 contém todos os valores entre 51 e 60 com os limites [51,60] inclusos no grupo;


e assim por diante ...

In [0]:

"""
    Através do overall do jogador retorne a classificação conforme a seguir:
    Overall -> classification
    -50     -> "Amador"
    51-60   -> "Ruim"
    61-70   -> "Ok"
    71-80   -> "Bom"
    81-90   -> "Ótimo"
    91+     -> "Lenda"
    
    I: int overall
    O: string
"""

# Dica utilize as clasulas when e otherwise
fut_players = (fut_players
               .select('player_id', 'overall')
               .withColumn('classification', F.when(F.col("overall") <= 50, "Amador")
                            .when(F.col("overall") <= 60, "Ruim")
                            .when(F.col("overall") <= 70, "Ok")
                            .when(F.col("overall") <= 80, "Bom")
                            .when(F.col("overall") <= 90, "Otimo")
                            .otherwise("Lenda")
                           )
               )

# Contar quantos jogadores há em cada classificação
fut_players.groupBy("classification").count().orderBy("count", ascending=False).display()

## Desafio — Montando o Time dos Sonhos do 🇧🇷

Ainda utilizando a base **`fut_players_data`**, imagine que você é um grande fã do jogo *FIFA*, e deseja montar o **Time dos Sonhos (Dream Team)** do **Brasil**, selecionando os **melhores jogadores por posição**, ou seja, aqueles com o **maior overall** dentro de cada grupo de posição.

Para isso, adote a **formação tática 4-4-2**, composta por:

- **1 Goleiro (GK)**  
- **4 Defensores (Defesa)**  
- **4 Meio-campistas (Meio)**  
- **2 Atacantes (Ataque)**  

### Objetivo
Criar um *DataFrame* com **11 linhas**, representando o **melhor jogador de cada posição dentro da formação 4-4-2**, com as seguintes colunas:

- `nationality` — nacionalidade do jogador  
- `position_group` — posição agrupada (Goleiro, Defesa, Meio, Ataque)  
- `player_name` — nome do jogador  
- `overall` — nota geral (overall)

---

### Agrupamento de posições
Para facilitar a análise, agrupe as posições originais da base conforme a tabela abaixo:

| **position_group** | **Posições incluídas (`position`)** | **Descrição** |
|:--------------------|:------------------------------------|:---------------|
| **Goleiro** | `GK` | Jogadores que atuam exclusivamente no gol. |
| **Defesa** | `CB`, `LB`, `RB`, `LWB`, `RWB` | Zagueiros e laterais (defensores). |
| **Meio** | `CM`, `CDM`, `CAM`, `LM`, `RM` | Meio-campistas centrais, volantes e meias ofensivos/laterais. |
| **Ataque** | `ST`, `CF`, `LW`, `RW`, `LF`, `RF` | Atacantes e pontas. |
| **Outros** | *(demais posições não classificadas)* | Jogadores fora do esquema tático principal (ex: cartas especiais). |

---

### 🏁 Entrega esperada
Seu *DataFrame final* deve retornar **11 jogadores**, representando o **Time dos Sonhos do Brasil (formação 4-4-2)**, conforme os critérios acima.

In [0]:
"""
A solução apresentada foi construída considerando o escopo e as regras específicas deste exercício. Em um cenário com uma base de dados maior, mais posições ou diferentes formações táticas, seria interessante utilizar tabelas auxiliares e join para armazenar essas regras de forma mais flexível e evitar que elas fiquem diretamente codificadas. A mesma abordagem poderia ser aplicada ao mapeamento de position para position_group, utilizando uma tabela de referência para essa conversão.
"""

fut_players = spark.read.csv(
    "/Workspace/Users/jhonatasousa288@gmail.com/rocket-lab-2026/solucoes/pratica-001/fut_players_data.csv", 
    header=True,
    inferSchema=True)

fut_players = fut_players.withColumn("position_group", 
    F.when(fut_players.position == "GK", "Goleiro")
    .when(F.col("position").isin("CB", "LB", "RB", "LWB", "RWB"), "Defesa")                                  
    .when(F.col("position").isin("CM", "CDM", "CAM", "LM", "RM"), "Meio")                                  
    .when(F.col("position").isin("ST", "CF", "LW", "RW", "LF", "RF"), "Ataque")                                  
    .otherwise("Outros")                                  
)

wGoleiros = Window.partitionBy("position_group").orderBy(F.col("overall").desc())

fut_players = fut_players.withColumn("rank", F.row_number().over(wGoleiros))

fut_players = fut_players.withColumn("max_players", 
    F.when(fut_players.position_group == "Goleiro", 1)
    .when(fut_players.position_group == "Defesa", 4)                                  
    .when(fut_players.position_group == "Meio", 4)                                  
    .when(fut_players.position_group == "Ataque", 2)                                                                 
)

# Vale aqui o comentario de que a solução aqui construida leva em consideração somente o q foi pedido,
# faria mais sentido para uma base de dados maior ou com mais variações de formação, fosse interessante
# criar uma tabela auxiliar e fazer um join a fim de chegar ao msm objetivo que o withColumn acima.
# A mesma ideia poderia se aplicar a criar uma tabela auxiliar de conversao de position -> position_group 


fut_players = fut_players.filter(F.col("rank") <= F.col("max_players"))

display(
    fut_players.select(
        "player_name",
        "position_group",
        "overall",
        "rank"
    )
)


### Desafio Bônus

Você deve ter notado que **Neymar** aparece tanto entre os melhores jogadores de **ataque** quanto do **meio-campo**.  
Isso acontece porque o dataset contém **múltiplas versões do mesmo jogador**, inclusive atuando em **outras posições**, o que é típico dos modos do *FIFA/Ultimate Team*.

O seu desafio agora é **refazer o exercício anterior**, garantindo que **cada jogador apareça apenas uma vez** no *DataFrame final*.

- Caso o jogador possua mais de uma versão (carta), **considere apenas aquela com o maior valor de `overall`**.  
- Em seguida, **reaplique a lógica da formação 4-4-2**, selecionando os melhores por grupo de posição.

---

### 🏁 Entrega Esperada
Seu *DataFrame final* deve retornar **11 jogadores únicos**, representando o **Dream Team do Brasil** na **formação tática 4-4-2**, **sem repetição de atletas**, conforme os critérios estabelecidos acima.


In [0]:
fut_players = spark.read.csv(
    "/Workspace/Users/jhonatasousa288@gmail.com/rocket-lab-2026/solucoes/pratica-001/fut_players_data.csv", 
    header=True,
    inferSchema=True)

fut_players = fut_players.withColumn("position_group", 
    F.when(fut_players.position == "GK", "Goleiro")
    .when(F.col("position").isin("CB", "LB", "RB", "LWB", "RWB"), "Defesa")                                  
    .when(F.col("position").isin("CM", "CDM", "CAM", "LM", "RM"), "Meio")                                  
    .when(F.col("position").isin("ST", "CF", "LW", "RW", "LF", "RF"), "Ataque")                                  
    .otherwise("Outros")                                  
)

wGoleiros = Window.partitionBy("position_group").orderBy(F.col("overall").desc())
wVersoes = Window.partitionBy("player_name").orderBy(F.col("overall").desc())

fut_players = fut_players.withColumn("version", F.row_number().over(wVersoes))

fut_players = fut_players.filter(F.col("version") == 1)

fut_players = fut_players.withColumn("rank", F.row_number().over(wGoleiros))

fut_players = fut_players.withColumn("max_players", 
    F.when(fut_players.position_group == "Goleiro", 1)
    .when(fut_players.position_group == "Defesa", 4)                                  
    .when(fut_players.position_group == "Meio", 4)                                  
    .when(fut_players.position_group == "Ataque", 2)                                                                 
)

fut_players = fut_players.filter(F.col("rank") <= F.col("max_players"))


display(
    fut_players.select(
        "player_name",
        "position_group",
        "overall",
        "rank"
    )
)


# Declaração de Inexistência de Plágio:

1. Eu sei que plágio é utilizar o trabalho de outra pessoa e apresentar como meu.
2. Eu sei que plágio é errado e declaro que este notebook foi feito por mim.
3. Tenho consciência de que a utilização do trabalho de terceiros é antiético e está sujeito a medidas administrativas.
4. Declaro também que não compartilhei e não compartilharei meu trabalho com o intuito de que seja copiado e submetido por outra pessoa.